# 05. Export Optimized JSON for Dashboard
This notebook creates lightweight JSON files that will be used in the Next.js + Vercel dashboard.

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "covid_cleaned.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "json"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(PROCESSED_PATH, parse_dates=["date"])
print("Data loaded:", df.shape)

Data loaded: (572427, 66)


In [2]:
# Latest record for each country
latest = df.sort_values("date").groupby("country").tail(1).copy()

# Keep only useful columns
latest_clean = latest[[
    "country", "continent", "code",
    "total_cases", "total_deaths",
    "total_cases_per_million", "total_deaths_per_million",
    "case_fatality_rate", "fully_vaccinated_rate",
    "population", "median_age", "gdp_per_capita", "life_expectancy"
]].copy()

# Convert to proper types
latest_clean = latest_clean.replace([np.inf, -np.inf], np.nan)
latest_clean = latest_clean.fillna(0)

# Save as JSON
latest_clean.to_json(OUTPUT_DIR / "latest.json", orient="records", indent=2)
print("✅ latest.json created")
print("Countries:", len(latest_clean))

✅ latest.json created
Countries: 239


In [3]:
global_daily = df.groupby("date").agg({
    "new_cases": "sum",
    "new_deaths": "sum",
    "new_cases_7day_avg": "sum",
    "new_deaths_7day_avg": "sum"
}).reset_index()

# Keep only needed columns and format date
global_daily["date"] = global_daily["date"].dt.strftime("%Y-%m-%d")
global_daily = global_daily.round(0)

global_daily.to_json(OUTPUT_DIR / "global_trend.json", orient="records", indent=2)
print("✅ global_trend.json created")
print("Days:", len(global_daily))

✅ global_trend.json created
Days: 2434


In [4]:
continent_summary = latest.groupby("continent").agg({
    "total_cases": "sum",
    "total_deaths": "sum",
    "population": "sum"
}).reset_index()

continent_summary = continent_summary.round(0)
continent_summary.to_json(OUTPUT_DIR / "continent_summary.json", orient="records", indent=2)
print("✅ continent_summary.json created")

✅ continent_summary.json created


In [5]:
# Get top 30 countries by total cases
top_countries = latest.nlargest(30, "total_cases")["country"].tolist()

# Filter only those countries
df_top = df[df["country"].isin(top_countries)][["date", "country", "new_cases_7day_avg", "new_deaths_7day_avg"]].copy()
df_top["date"] = df_top["date"].dt.strftime("%Y-%m-%d")
df_top = df_top.round(0)

# Save
df_top.to_json(OUTPUT_DIR / "top30_trend.json", orient="records", indent=2)
print("✅ top30_trend.json created")

✅ top30_trend.json created


In [ ]:
print("\n" + "="*50)
print("JSON FILES CREATED SUCCESSFULLY")
print("="*50)

for file in OUTPUT_DIR.glob("*.json"):
    size_kb = file.stat().st_size / 1024
    print(f"{file.name:<25} → {size_kb:.1f} KB")

print("\nFiles are ready in: data/json/")


JSON FILES CREATED SUCCESSFULLY
continent_summary.json    → 0.7 KB
global_trend.json         → 352.9 KB
latest.json               → 98.0 KB
top30_trend.json          → 8671.7 KB

Files are ready in: data/json/


In [2]:
import pandas as pd
from pathlib import Path

# Setup paths
PROJECT_ROOT = Path.cwd().parent
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "covid_cleaned.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "json"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load data again
df = pd.read_csv(PROCESSED_PATH, parse_dates=["date"])
print("Data loaded:", df.shape)

# Create Yearly Deaths
df['year'] = df['date'].dt.year

yearly_deaths = (
    df.groupby(['country', 'continent', 'year'])['new_deaths']
    .sum()
    .reset_index()
    .rename(columns={'new_deaths': 'deaths'})
)

# Keep only years with deaths
yearly_deaths = yearly_deaths[yearly_deaths['deaths'] > 0]

# Save
yearly_deaths.to_json(OUTPUT_DIR / "yearly_deaths.json", orient="records", indent=2)

print("✅ yearly_deaths.json created successfully!")
print("Total records:", len(yearly_deaths))
print(yearly_deaths.head(10))

Data loaded: (572427, 66)
✅ yearly_deaths.json created successfully!
Total records: 929
        country continent  year  deaths
0   Afghanistan      Asia  2020  2189.0
1   Afghanistan      Asia  2021  5167.0
2   Afghanistan      Asia  2022   493.0
3   Afghanistan      Asia  2023   124.0
4   Afghanistan      Asia  2024    25.0
7       Albania    Europe  2020  1170.0
8       Albania    Europe  2021  2042.0
9       Albania    Europe  2022   384.0
10      Albania    Europe  2023     8.0
11      Albania    Europe  2024     4.0
